# Lesson 4.5: The Bootstrap (Bonus)

Bootstrap one small sample of five heart rates: get a standard error and a 95% interval without any formula, then compare with the t-interval.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. The sample

Five resting heart rates (bpm) from the lesson. The classic formula SE = s / sqrt(n) is our benchmark.

In [1]:
import numpy as np
from scipy import stats

data = np.array([62, 70, 68, 75, 65])
n = len(data)
se = data.std(ddof=1) / np.sqrt(n)

print(f'mean = {data.mean():.1f}')
print(f's    = {data.std(ddof=1):.2f}')
print(f'classic SE = {se:.2f}')

mean = 68.0
s    = 4.95
classic SE = 2.21


## 2. Resample 5,000 times

Each resample draws 5 values from our data with replacement (the same person can appear twice), then takes the mean. We keep all 5,000 means.

In [2]:
rng = np.random.default_rng(2025)
resamples = rng.choice(data, size=(5000, n), replace=True)
boot_means = resamples.mean(axis=1)

print('one resample:', resamples[0], ' mean =', resamples[0].mean())
print(f'average of boot means = {boot_means.mean():.2f}')

one resample: [68 65 65 70 65]  mean = 66.6
average of boot means = 68.04


## 3. Bootstrap SE and interval

The SD of the bootstrap means is the bootstrap standard error. The 2.5th and 97.5th percentiles of the means form a 95% percentile interval.

In [3]:
boot_se = boot_means.std(ddof=1)
low, high = np.percentile(boot_means, [2.5, 97.5])

print(f'bootstrap SE = {boot_se:.2f}')
print(f'95% percentile interval = ({low:.1f}, {high:.1f})')

bootstrap SE = 1.99
95% percentile interval = (64.4, 72.0)


## 4. Compare with the t-interval

With only five values the t-interval has to be wide to be honest. The bootstrap interval is narrower, because resampling cannot invent information a tiny sample does not contain.

In [4]:
t_low, t_high = stats.t.interval(0.95, df=n - 1, loc=data.mean(), scale=se)

print(f't-interval        = ({t_low:.1f}, {t_high:.1f})')
print(f'bootstrap interval = ({low:.1f}, {high:.1f})')
print(f'widths: t = {t_high - t_low:.1f}, bootstrap = {high - low:.1f}')

t-interval        = (61.9, 74.1)
bootstrap interval = (64.4, 72.0)
widths: t = 12.3, bootstrap = 7.6


## Try this

- Change 5000 resamples to 500 and see how much the bootstrap SE changes between seeds.
- Add more heart rates (say ten values) and predict whether the bootstrap and t-intervals move closer together.
- Replace the mean with the median (np.median(resamples, axis=1)) and look at its bootstrap SE.

## The same idea in R, SPSS and Stata

**R**

```r
hr <- c(62, 70, 68, 75, 65)
set.seed(2025)
boot_means <- replicate(5000, mean(sample(hr, replace = TRUE)))
sd(boot_means)
quantile(boot_means, c(0.025, 0.975))
t.test(hr)$conf.int
```

**SPSS** (Analyze > Descriptive Statistics > Explore, then press the Bootstrap button and tick Perform bootstrapping (Number of samples 5000, Percentile))

```
DATA LIST FREE / hr.
BEGIN DATA
62 70 68 75 65
END DATA.
BOOTSTRAP
  /SAMPLING METHOD=SIMPLE
  /VARIABLES INPUT=hr
  /CRITERIA CILEVEL=95 CITYPE=PERCENTILE NSAMPLES=5000
  /MISSING USER=EXCLUDE.
EXAMINE VARIABLES=hr
  /PLOT=NONE
  /STATISTICS=DESCRIPTIVES.
```

**Stata**

```
clear
input hr
62
70
68
75
65
end
bootstrap mean = r(mean), reps(5000) seed(2025): summarize hr
estat bootstrap, percentile
ci means hr
```